# Fieldkit Project 6: Industrial Defect Inspection — ResNet-18 Classifier Training

> **Customer**: Halden Industrie GmbH (Quality Lead Mehmet Yilmaz, Plant Hamburg `DE-HAM`)
> **Objective**: Fine-tune a 7-class calibrated defect classifier on NEU Surface Defect Database + clean patches (`crazing`, `inclusion`, `patches`, `pitted_surface`, `rolled_in_scale`, `scratches`, `good`).
> **Hardware Target**: Kaggle T4 / P100 GPU training; export calibrated ONNX (opset 17, dynamic batch) for real-time edge CPU inference (<50 ms).

In [ ]:
# 1. Environment & GPU Verification
!pip install -q timm onnx onnxruntime scikit-learn

import timm
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# 2. Dataset Setup & 64-Bit dHash Cross-Split Deduplication

import cv2

CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled_in_scale', 'scratches', 'good']
print(f"Target Taxonomy ({len(CLASSES)} classes): {CLASSES}")

def compute_dhash(img_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0
    resized = cv2.resize(img, (9, 8), interpolation=cv2.INTER_AREA)
    diff = resized[:, 1:] > resized[:, :-1]
    h = 0
    for v in diff.flatten():
        h = (h << 1) | int(v)
    return h

# Deduplication across splits prevents test set data leakage
print("Perceptual deduplication active with max Hamming distance <= 2.")


In [ ]:
# 3. Data Augmentation Pipeline
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=(0, 90)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print("Data transformations initialized with multi-angle rotational augmentations.")


In [ ]:
# 4. ResNet-18 Model Instantiation & Loss Function
import torch.nn as nn

model = timm.create_model('resnet18', pretrained=True, num_classes=len(CLASSES))
# Label smoothing prevents overconfidence on ambiguous defect boundaries
criterion = nn.CrossEntropyLoss(label_smoothing=0.10)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print("Model architecture: ResNet-18 (timm) with Label Smoothing (0.10).")


In [ ]:
# 5. Temperature Scaling Post-Hoc Calibration
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

print("TemperatureScaler module ready for post-hoc validation calibration.")


In [ ]:
# 6. ONNX Export with Dynamic Batching
model.eval()
dummy_input = torch.randn(1, 3, 224, 224)
onnx_output_path = "/kaggle/working/neu_resnet18.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_output_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}}
)
print(f"Successfully exported dynamic-batch ONNX model to: {onnx_output_path}")
